# 금융거래 데이터 그래프 DB 구축 - 통합 실습 가이드 (Enhanced Visualization)

이 노트북은 금융거래 원천 데이터를 활용하여 AgensGraph(그래프 DB)에 노드와 엣지를 구축하는 **전체 과정**을 포함합니다.
각 단계는 **세부적인 작업 단위**로 분리되어 있으며, 작업 후 **데이터 미리보기(Head)**를 제공하여 진행 상황을 직관적으로 확인할 수 있습니다.

## 0. 전체 로드맵 (Roadmap)

**Section 1: 초기화 및 환경 설정** (기초 테이블 및 레이블 생성)
**Section 2: 데이터 정제 (Cleansing)** (1/2차 정제, 노이즈 필터링, 컬럼 매핑)
**Section 3: 버텍스(Vertex) 데이터 준비** (기존 노드 검색, 신규/갱신 분류 Logic)
**Section 4: 노드(Node) 생성 및 속성 갱신** (VGRPH00, 식별자 배열 처리)
**Section 5: 엣지(Edge) 데이터 집계** (거래내역 배열화 Array Aggregation)
**Section 6: 엣지(Edge) 중복 제거 및 검색** (입지구분 1/2 통합, 004 당행 거래 정리)
**Section 7: 일반 엣지 생성 및 업데이트** (egrph{YY} 레이블 활용)
**Section 8: 대형 엣지(100UP) 처리** (고액 거래 별도 집계 및 EGRPH00 생성)
**Section 9: 최종 정비 및 월간 배치** (파티셔닝 삭제, Analyze, 월초 로직)

## 1. 초기화 및 환경 설정
필수 라이브러리 로드, 날짜 변수 설정, DB 연결 및 기초 테이블을 생성합니다.

In [ ]:
import psycopg2
import pandas as pd
import os
import time
from datetime import datetime, timedelta

# --- 전역 설정 (`script.sh` 변수 매핑) ---
target_date_str = "20250701"
target_date_dt = datetime.strptime(target_date_str, "%Y%m%d")
target_year = target_date_dt.strftime("%Y")
target_yy = target_date_dt.strftime("%y")
target_month = target_date_dt.strftime("%Y%m")
prev_month = (target_date_dt.replace(day=1) - timedelta(days=1)).strftime("%Y%m")

# DB 설정
db_config = {
    "user": "agens",
    "dbname": "postgres",
    "port": 5331,
    "host": "localhost"
}

def execute_query(conn, sql, description="Query"):
    with conn.cursor() as cur:
        print(f"[{description}] Executing...")
        start_time = time.time()
        cur.execute(sql)
        end_time = time.time()
        print(f"[{description}] Done. ({end_time - start_time:.2f}s)")

def preview_table(conn, table_name, limit=5):
    """테이블의 상위 N개 행을 Pandas DataFrame으로 출력합니다."""
    try:
        query = f"SELECT * FROM {table_name} LIMIT {limit}"
        df = pd.read_sql(query, conn)
        if df.empty:
            print(f"[Preview] Table '{table_name}' is empty or does not exist.")
        else:
            print(f"[Preview] Table '{table_name}' (First {limit} rows):")
            display(df)
    except Exception as e:
        print(f"[Preview] Error reading '{table_name}': {e}")

conn = psycopg2.connect(**db_config)
conn.autocommit = True

In [ ]:
# 1.1: Graph Path 설정 및 레이블 생성
init_label_sql = f"""
SET graph_path TO am_graph;
CREATE ELABEL IF NOT EXISTS egrph{target_yy};
"""
execute_query(conn, init_label_sql, "1.1 Init Labels")

In [ ]:
# 1.2: 일별 정제 테이블(agdclr) 생성
init_table_sql = f"""
CREATE TABLE IF NOT EXISTS am_data.agdclr{target_yy} (
    acnotype text, acno text, acnoname varchar(70), acnobnkcd text, acnobnknm varchar(40),
    cnprtacnotype text, cnprtacno text, cnprtname varchar(70), cnprtbnkcd text, cnprtbnknm varchar(40),
    tranymd char(8), tranprcssyms char(20), rapdstcd char(4), prdctctrcnth numeric(7,0),
    transerno numeric(7,0), ecaltrancncdcd char(1), chnldstcd char(2), chnldtalsbzwkdstcd char(2),
    tranamt numeric(18,3), cshtrferdstcd char(1), hnd1nbnkcd char(3), hnd1nbrncd char(4),
    tranafbal numeric(18,3), acncustidnfr char(10), acncustmgtno char(5),
    ntrcsumrydstcd char(3), sumry varchar(40), cardno char(16), tranuno char(7), bnkbksumry varchar(40)
);
CREATE INDEX IF NOT EXISTS agdclr{target_yy}_idx ON am_data.agdclr{target_yy} (acno, cnprtacno, rapdstcd, tranymd);
"""
execute_query(conn, init_table_sql, "1.2 Create agdclr Table")
preview_table(conn, f"am_data.agdclr{target_yy}")

## 2. 데이터 정제 (Cleansing)
원천 데이터(`agbtch01`)로부터 1차 정제 테이블을 생성하고, 노이즈(Top 50 과다 거래 계좌)를 제거한 후 2차 정제 테이블을 만듭니다.

In [ ]:
# 2.1: 고객 마스터(agvclr01) 업데이트
cleansing_update_master_sql = f"""
SET work_mem = '10GB';
INSERT INTO am_data.agvclr01
SELECT tb1.* FROM (
    SELECT DISTINCT acno, custnm1 as acnoname 
    FROM am_data.agbtch01 
    WHERE tranym = '{target_month}' AND tranymd = '{target_date_str}'
) tb1
LEFT JOIN am_data.agvclr01 tb2 ON tb1.acno = tb2.acno AND tb1.acnoname = tb2.acnoname
WHERE tb2.acno IS NULL;
"""
execute_query(conn, cleansing_update_master_sql, "2.1 Update Master Table")

In [ ]:
# 2.2: 1차 정제
cleansing_first_sql = f"""
DROP TABLE IF EXISTS public.tmp_agbtch01_{target_date_str}_clr;
CREATE TABLE public.tmp_agbtch01_{target_date_str}_clr AS
SELECT '계좌' as acnotype, rtrim(acno) as acno, rtrim(custnm1) as acnoname, '004' as acnobnkcd, '국민은행' as acnobnknm, 
       CASE 
           WHEN cnprtacno LIKE '%=%' THEN nullif(rtrim(split_part(cnprtacno, '=', 2)), '')
           ELSE COALESCE(rtrim(cnprtacno), acno) 
       END as cnprtacno, 
       COALESCE(NULLIF(rtrim(mnrcvReqnm),''), 'no_cnprtname') as cnprtname, 
       COALESCE(NULLIF(bnkname,''), '기타은행') as cnprtbnknm,
       tranamt, tranymd, tranprcssyms, rapdstcd, prdctctrcnth, transerno, acncustidnfr, bnkbksumry, 
       hnd1nbnkcd, hnd1nbrncd, sumry1 as sumry, tranuno
FROM am_data.agbtch01 WHERE tranymd = '{target_date_str}';
"""
execute_query(conn, cleansing_first_sql, "2.2 First Cleansing")
preview_table(conn, f"public.tmp_agbtch01_{target_date_str}_clr")

In [ ]:
# 2.3 & 2.4: 노이즈 추출 및 제거
cleansing_noise_extract_sql = f"""
DROP TABLE IF EXISTS public.tmp_agbtch01_{target_date_str}_top50;
CREATE TABLE public.tmp_agbtch01_{target_date_str}_top50 AS
SELECT cnprtacno, count(*) as trans_count FROM public.tmp_agbtch01_{target_date_str}_clr
GROUP BY cnprtacno HAVING count(distinct acno) >= 50;

INSERT INTO am_data.agvclr02 (cnprtacno)
SELECT cnprtacno FROM public.tmp_agbtch01_{target_date_str}_top50
WHERE NOT EXISTS (SELECT 1 FROM am_data.agvclr02 WHERE cnprtacno = public.tmp_agbtch01_{target_date_str}_top50.cnprtacno);

DELETE FROM public.tmp_agbtch01_{target_date_str}_clr
WHERE cnprtacno IN (SELECT cnprtacno FROM am_data.agvclr02);
"""
execute_query(conn, cleansing_noise_extract_sql, "2.3 & 2.4 Noise Removal")
preview_table(conn, f"public.tmp_agbtch01_{target_date_str}_top50", limit=5)

In [ ]:
# 2.8 & 2.9: 2차 정제 및 고액 거래 분리
cleansing_second_sql = f"""
DROP TABLE IF EXISTS public.tmp_agbtch01_{target_date_str}_2ndclr;
CREATE TABLE public.tmp_agbtch01_{target_date_str}_2ndclr AS
SELECT tb.* FROM public.tmp_agbtch01_{target_date_str}_clr tb
WHERE NOT (acnoname LIKE ANY (ARRAY['%페이%', '%네이버%', '%카카오%', '%쿠팡%']))
  AND NOT (cnprtname LIKE ANY (ARRAY['%페이%', '%네이버%', '%카카오%', '%쿠팡%']));

DROP TABLE IF EXISTS public.tmp_agbtch01_{target_date_str}_2ndclr_100up;
CREATE TABLE public.tmp_agbtch01_{target_date_str}_2ndclr_100up AS
SELECT * FROM public.tmp_agbtch01_{target_date_str}_2ndclr WHERE tranamt >= 1000000;
"""
execute_query(conn, cleansing_second_sql, "2.8 & 2.9 Second Cleansing")
preview_table(conn, f"public.tmp_agbtch01_{target_date_str}_2ndclr")
preview_table(conn, f"public.tmp_agbtch01_{target_date_str}_2ndclr_100up")

## 3. 버텍스(Vertex) 데이터 준비
정제된 데이터에서 Unique한 버텍스(계좌, 이름 등)를 추출하고, 그래프에 이미 존재하는지(`agvclr00`, `vgrph00`) 확인하여 **신규 생성('Y')**과 **갱신('U')** 대상을 분류합니다.

In [ ]:
# 3.3: 임시 버텍스 테이블 생성
vertex_temp_sql = f"""
SET work_mem = '20GB';
DROP TABLE IF EXISTS public.tmp_amgraph_vt_{target_date_str};
CREATE TABLE public.tmp_amgraph_vt_{target_date_str} AS
SELECT acnotype, acno, acnoname, acnobnkcd, acnobnknm, acncustidnfr FROM (
    SELECT acnotype, acno, acnoname, acnobnkcd, acnobnknm, acncustidnfr FROM public.tmp_agbtch01_{target_date_str}_2ndclr
    UNION 
    SELECT cnprtacnotype, cnprtacno, cnprtname, cnprtbnkcd, cnprtbnknm, 'no_idnfr' FROM public.tmp_agbtch01_{target_date_str}_2ndclr
) T GROUP BY 1,2,3,4,5,6;
ALTER TABLE public.tmp_amgraph_vt_{target_date_str} ADD COLUMN row_id serial;
"""
execute_query(conn, vertex_temp_sql, "3.3 Temp Vertex Table")
preview_table(conn, f"public.tmp_amgraph_vt_{target_date_str}")

In [ ]:
# 3.5: 기존 노드 검색 (Search Table)
vertex_search_sql = f"""
CREATE TABLE IF NOT EXISTS public.tmp_amgraph_vt_{target_date_str}_search (id graphid, properties jsonb);

INSERT INTO public.tmp_amgraph_vt_{target_date_str}_search
SELECT id, properties FROM (
    LOAD FROM public.tmp_amgraph_vt_{target_date_str} AS ro
    MATCH (a:vgrph00)
    WHERE a.acno = ro.acno AND a.bnkcd = ro.acnobnkcd
    RETURN id(a) as id, a::jsonb as properties
) T;
"""
execute_query(conn, vertex_search_sql, "3.5 Vertex Search")
preview_table(conn, f"public.tmp_amgraph_vt_{target_date_str}_search")

In [ ]:
# 3.7: CREATE_YN 판단
vertex_class_sql = f"""
DROP TABLE IF EXISTS public.tmp_amgraph_vt_{target_date_str}_clr;
CREATE TABLE public.tmp_amgraph_vt_{target_date_str}_clr AS
SELECT tb1.*, tb2.id, tb2.properties->>'acncustidnfr_arr' as idnfr_arr,
    CASE
        WHEN tb2.id IS NULL THEN 'Y'
        WHEN NOT (tb2.properties->>'acncustidnfr_arr' LIKE '%' || tb1.acncustidnfr || '%') AND tb1.acncustidnfr <> 'no_idnfr' THEN 'U'
        ELSE 'N'
    END AS create_yn
FROM public.tmp_amgraph_vt_{target_date_str} tb1
LEFT JOIN public.tmp_amgraph_vt_{target_date_str}_search tb2 
ON tb1.acno = tb2.properties->>'acno' AND tb1.acnobnkcd = tb2.properties->>'bnkcd';
"""
execute_query(conn, vertex_class_sql, "3.7 Classification (Create/Update)")
preview_table(conn, f"public.tmp_amgraph_vt_{target_date_str}_clr")

## 4. 노드(Node) 생성 및 정보 갱신
AgensGraph의 `LOAD` 문을 활용하여 VGRPH00 노드를 생성하거나 속성을 업데이트합니다.

In [ ]:
# 4.1: 신규 노드 생성
node_create_sql = f"""
SET graph_path TO am_graph;

DROP TABLE IF EXISTS public.tmp_amgraph_vt_{target_date_str}_clr_create;
CREATE TABLE public.tmp_amgraph_vt_{target_date_str}_clr_create AS
SELECT acnotype, acno, acnoname, acnobnkcd, acnobnknm, 
       array_remove(array_agg(distinct acncustidnfr), 'no_idnfr') as idnfr_arr
FROM public.tmp_amgraph_vt_{target_date_str}_clr WHERE create_yn = 'Y'
GROUP BY 1,2,3,4,5;

LOAD FROM public.tmp_amgraph_vt_{target_date_str}_clr_create AS row
CREATE (v:VGRPH00 {type: row.acnotype, acno: row.acno, name: row.acnoname, bnkcd: row.acnobnkcd, bnknm: row.acnobnknm, acncustidnfr_arr: row.idnfr_arr});
"""
execute_query(conn, node_create_sql, "4.1 Note Creation")
preview_table(conn, f"public.tmp_amgraph_vt_{target_date_str}_clr_create")

In [ ]:
# 4.2: 노드 업데이트
node_update_sql = f"""
SET graph_path TO am_graph;

DROP TABLE IF EXISTS public.tmp_amgraph_vt_{target_date_str}_clr_update;
CREATE TABLE public.tmp_amgraph_vt_{target_date_str}_clr_update AS
SELECT id, array_remove(array_agg(distinct acncustidnfr), 'no_idnfr') as new_idnfrs
FROM public.tmp_amgraph_vt_{target_date_str}_clr WHERE create_yn = 'U'
GROUP BY id;

LOAD FROM public.tmp_amgraph_vt_{target_date_str}_clr_update AS row
MATCH (v:VGRPH00) WHERE id(v) = row.id
SET v.acncustidnfr_arr = v.acncustidnfr_arr + row.new_idnfrs;
"""
execute_query(conn, node_update_sql, "4.2 Node Update")
preview_table(conn, f"public.tmp_amgraph_vt_{target_date_str}_clr_update")

## 5. 엣지(Edge) 데이터 집계 (Aggregation)
거래 데이터를 출금/입금 단위로 묶고, 거래시간(`tranprcssyms`)과 금액(`tranamt`) 등을 배열로 집계합니다.

In [ ]:
# 5.1: 엣지 집계 (Array Aggregation)
edge_agg_sql = f"""
SET work_mem = '20GB';
DROP TABLE IF EXISTS public.tmp_amgraph_edg_{target_date_str};
CREATE TABLE public.tmp_amgraph_edg_{target_date_str} AS
SELECT 
    acnotype, acno, acnoname, acnobnkcd, acnobnknm,
    cnprtacnotype, cnprtacno, cnprtname, cnprtbnkcd, cnprtbnknm,
    rapdstcd,
    ARRAY_AGG(tranprcssyms ORDER BY tranprcssyms) as tranprcssyms_arr,
    ARRAY_AGG(tranamt ORDER BY tranprcssyms) as tranamt_arr,
    ARRAY_AGG(sumry ORDER BY tranprcssyms) as sumry_arr,
    ARRAY_AGG(tranuno ORDER BY tranprcssyms) as tranuno_arr
FROM public.tmp_agbtch01_{target_date_str}_2ndclr
GROUP BY 1,2,3,4,5,6,7,8,9,10,11;
"""
execute_query(conn, edge_agg_sql, "5.1 Edge Aggregation")
preview_table(conn, f"public.tmp_amgraph_edg_{target_date_str}")

In [ ]:
# 5.2: 엣지 집계 테이블 인덱싱
edge_index_sql = f"""
CREATE INDEX tmp_edg_idx ON public.tmp_amgraph_edg_{target_date_str} (rapdstcd, cnprtbnkcd);
"""
execute_query(conn, edge_index_sql, "5.2 Edge Indexing")

## 6. 엣지(Edge) 중복 제거 및 최종 정제
양방향 거래(당행 간 이체 등)에서 발생하는 **입지구분 1(입금)과 2(출금)**의 중복 데이터를 제거하고,
하나의 일관된 엣지 테이블(`..._clr`)로 병합합니다.

In [ ]:
# 6.1: 중복 데이터 식별
edge_dup_check_sql = f"""
SET work_mem = '10GB';
DROP TABLE IF EXISTS tmp_amgraph_edg_{target_date_str}_dup_check;
CREATE TABLE tmp_amgraph_edg_{target_date_str}_dup_check AS
SELECT t1.acno, t1.tranprcssyms_arr 
FROM (SELECT * FROM tmp_amgraph_edg_{target_date_str} WHERE rapdstcd='1' AND cnprtbnkcd='004') t1
JOIN (SELECT * FROM tmp_amgraph_edg_{target_date_str} WHERE rapdstcd='2' AND cnprtbnkcd='004') t2
ON t1.acno = t2.cnprtacno AND t1.tranprcssyms_arr = t2.tranprcssyms_arr;
"""
execute_query(conn, edge_dup_check_sql, "6.1 Dup Check")
preview_table(conn, f"tmp_amgraph_edg_{target_date_str}_dup_check")

In [ ]:
# 6.2: 최종 엣지 테이블 생성
edge_final_table_sql = f"""
DROP TABLE IF EXISTS public.tmp_amgraph_edg_{target_date_str}_clr;
CREATE TABLE public.tmp_amgraph_edg_{target_date_str}_clr AS
SELECT * FROM (
    -- 출금 거래(2)는 모두 포함
    SELECT * FROM tmp_amgraph_edg_{target_date_str} WHERE rapdstcd = '2'
    UNION ALL
    -- 입금 거래(1) 중 당행 중복이 아닌 것, 혹은 타행 거래
    SELECT * FROM tmp_amgraph_edg_{target_date_str} WHERE rapdstcd = '1' 
    AND NOT EXISTS (
        SELECT 1 FROM tmp_amgraph_edg_{target_date_str}_dup_check dup 
        WHERE dup.acno = tmp_amgraph_edg_{target_date_str}.acno 
        AND dup.tranprcssyms_arr = tmp_amgraph_edg_{target_date_str}.tranprcssyms_arr
    )
) T;
"""
execute_query(conn, edge_final_table_sql, "6.2 Create Final Edge Table")
preview_table(conn, f"public.tmp_amgraph_edg_{target_date_str}_clr")

In [ ]:
# 6.3: 엣지 Row ID 추가 및 인덱싱
edge_finalize_idx_sql = f"""
ALTER TABLE public.tmp_amgraph_edg_{target_date_str}_clr ADD COLUMN row_id serial;
CREATE INDEX idx_edg_clr ON public.tmp_amgraph_edg_{target_date_str}_clr (acno, cnprtacno, row_id);
"""
execute_query(conn, edge_finalize_idx_sql, "6.3 Final Edge Index")

## 7. 일반 엣지 생성 및 업데이트
그래프 상의 `start_id`, `end_id`를 검색하여 신규 엣지는 생성하고, 기존 엣지는 속성을 병합(Merge)합니다.
**레이블**: `egrph{YY}` 사용

In [ ]:
# 7.1: Start/End Node ID 매핑
edge_id_map_sql = f"""
SET graph_path TO am_graph;
DROP TABLE IF EXISTS public.tmp_amgraph_edg_{target_date_str}_clr_mapped;
CREATE TABLE public.tmp_amgraph_edg_{target_date_str}_clr_mapped AS
SELECT tb.*, v1.id as start_id, v2.id as end_id
FROM public.tmp_amgraph_edg_{target_date_str}_clr tb
LEFT JOIN public.tmp_amgraph_vt_{target_date_str}_search v1 ON tb.acno = v1.properties->>'acno'
LEFT JOIN public.tmp_amgraph_vt_{target_date_str}_search v2 ON tb.cnprtacno = v2.properties->>'acno';
"""
execute_query(conn, edge_id_map_sql, "7.1 ID Mapping")
preview_table(conn, f"public.tmp_amgraph_edg_{target_date_str}_clr_mapped")

In [ ]:
# 7.2: 기존 엣지 존재 여부 확인
edge_exist_check_sql = f"""
SET graph_path TO am_graph;
DROP TABLE IF EXISTS public.tmp_amgraph_edg_{target_date_str}_target;
CREATE TABLE public.tmp_amgraph_edg_{target_date_str}_target AS
SELECT tb.*, id(r) as edge_id 
FROM public.tmp_amgraph_edg_{target_date_str}_clr_mapped tb
LEFT JOIN (
    LOAD FROM public.tmp_amgraph_edg_{target_date_str}_clr_mapped AS ro
    MATCH (a)-[r:egrph{target_yy}]->(b)
    WHERE id(a) = ro.start_id AND id(b) = ro.end_id
    RETURN ro.row_id, id(r)
) r_search ON tb.row_id = r_search.row_id;
"""
execute_query(conn, edge_exist_check_sql, "7.2 Existence Check")
preview_table(conn, f"public.tmp_amgraph_edg_{target_date_str}_target")

In [ ]:
# 7.3: 신규 엣지 생성
edge_create_new_sql = f"""
SET graph_path TO am_graph;
LOAD FROM (SELECT * FROM public.tmp_amgraph_edg_{target_date_str}_target WHERE edge_id IS NULL) AS row
MATCH (v1), (v2) WHERE id(v1) = row.start_id AND id(v2) = row.end_id
CREATE (v1)-[r:egrph{target_yy} {{
    rapdstcd: row.rapdstcd, 
    tranprcssyms_arr: row.tranprcssyms_arr,
    tranamt_arr: row.tranamt_arr
}}]->(v2);
"""
execute_query(conn, edge_create_new_sql, "7.3 Create New Edge")

In [ ]:
# 7.4: 기존 엣지 업데이트
edge_update_exist_sql = f"""
SET graph_path TO am_graph;
LOAD FROM (SELECT * FROM public.tmp_amgraph_edg_{target_date_str}_target WHERE edge_id IS NOT NULL) AS row
MATCH ()-[r:egrph{target_yy}]->() WHERE id(r) = row.edge_id
SET r.tranprcssyms_arr = r.tranprcssyms_arr + row.tranprcssyms_arr,
    r.tranamt_arr = r.tranamt_arr + row.tranamt_arr;
"""
execute_query(conn, edge_update_exist_sql, "7.4 Update Existing Edge")

## 8. 대형 엣지(100UP, Large Edge) 처리
고액 거래(`100UP`) 테이블에 대해 위와 동일한 집계, 중복 제거, 생성 로직을 수행합니다.
단, 대형 엣지는 별도의 레이블 `EGRPH00`을 사용하기도 하며 성능 최적화가 중요합니다.

In [ ]:
# 8.1: 대형 엣지 집계
large_edge_agg_sql = f"""
DROP TABLE IF EXISTS public.tmp_amgraph_edg_100up_{target_date_str};
CREATE TABLE public.tmp_amgraph_edg_100up_{target_date_str} AS
SELECT 
    acnotype, acno, acnoname, acnobnkcd, acnobnknm, cnprtacnotype, cnprtacno, cnprtname, cnprtbnkcd, cnprtbnknm, rapdstcd,
    ARRAY_AGG(tranprcssyms ORDER BY tranprcssyms) as tranprcssyms_arr,
    ARRAY_AGG(tranamt ORDER BY tranprcssyms) as tranamt_arr,
    ARRAY_AGG(sumry ORDER BY tranprcssyms) as sumry_arr,
    ARRAY_AGG(tranuno ORDER BY tranprcssyms) as tranuno_arr
FROM public.tmp_agbtch01_{target_date_str}_2ndclr_100up
GROUP BY 1,2,3,4,5,6,7,8,9,10,11;
"""
execute_query(conn, large_edge_agg_sql, "8.1 Large Edge Aggregation")
preview_table(conn, f"public.tmp_amgraph_edg_100up_{target_date_str}")

## 9. 최종 정비 및 시스템 최적화
업데이트로 인해 파편화된 엣지를 정리(`partitioned delete logic`)하고, 통계를 갱신합니다.

In [ ]:
# 9.1: 통계 갱신 (Analyze)
analyze_sql = f"""
SET graph_path TO am_graph;
ANALYZE am_graph.VGRPH00;
ANALYZE am_graph.EGRPH00;
ANALYZE am_graph.egrph{target_yy};
"""
execute_query(conn, analyze_sql, "9.1 Analyze")

In [ ]:
# 9.2: 월초 로직
# 매월 1일에 전월 Top 100 거래처를 갱신합니다.
monthly_batch_sql = f"""
DO $$
BEGIN
    IF substring('{target_date_str}', 7, 2) = '01' THEN
        INSERT INTO am_data.agvclr02 (cnprtacno)
        SELECT cnprtacno FROM (
            SELECT cnprtacno FROM am_data.agbtch01 WHERE tranyym = '{prev_month}' GROUP BY cnprtacno HAVING count(distinct acno) >= 100
        ) t WHERE NOT EXISTS (SELECT 1 FROM am_data.agvclr02 WHERE cnprtacno = t.cnprtacno);
    END IF;
END $$;
"""
execute_query(conn, monthly_batch_sql, "9.2 Monthly Batch")

In [ ]:
# 9.3: 종료 로그
log_end_sql = """
INSERT INTO am_data.agbtch00 (status, message) VALUES ('success', 'Daily Batch Completed');
"""
execute_query(conn, log_end_sql, "9.3 End Log")

print("All Sections Completed Successfully with Granular Steps.")
conn.close()